In [ ]:
# Visual-Inertial Bundle Adjustment: Math to Code Guide

This document covers the mathematics behind VI-BA and maps every equation to
the corresponding code in this repository.  Read it alongside the two reference
books already in the repo root (`slambook-en.pdf` and `SLAM Handbook.pdf`).

---

## Reference map

| Topic | Primary source | Where in this repo |
|---|---|---|
| IMU preintegration on manifold | Forster et al. TRO 2017 §III–IV | `crates/kornia-sensors/src/imu.rs` |
| VI-BA residuals and Jacobians | Forster et al. TRO 2017 Table I; ORB-SLAM3 Appendix A | `crates/kornia-slam/src/ba_schur.rs` |
| Schur-complement BA | Triggs et al. 1999 §5; slambook-en Ch. 9 | `ba_schur.rs` (existing visual BA) |
| SO(3) / SE(3) Lie theory | Joan Solà "A micro Lie theory" (arXiv 1812.01537) §6–7 | `kornia-algebra` |
| Covariance propagation | Forster et al. TRO 2017 §IV-B | `imu.rs::integrate` |

### Where to read the derivations

- **Forster et al. TRO 2017** — "On-Manifold Preintegration for Real-Time
  Visual–Inertial Odometry".  This is the canonical source for the
  preintegration model, the 9-DOF residual, and the bias Jacobians.
  - §III: manifold preintegration (what `PreintegratedImu::integrate` does)
  - §IV-A: residuals `r_R, r_v, r_p` (equations 21–23)
  - Table I: all analytical Jacobians used in the normal equations
  - §IV-B: covariance propagation (`A`, `B`, `N` matrices in `imu.rs:178–215`)

- **ORB-SLAM3 paper** — Campos et al. TPAMI 2021.
  - Appendix A: step-by-step Jacobian derivation in the world-frame
    right-perturbation convention (the cleanest reference for implementation).
  - §IV: how VI-BA fits into the full system (local window, marginalisation).

- **SLAM Handbook.pdf** (in repo root)
  - Part II covers factor-graph BA, information matrices, and the Schur trick
    in a unified notation.

- **slambook-en.pdf** (in repo root)
  - Chapter 9: Schur complement, marginalisation, bundle adjustment.
  - Chapter 4: SO(3) / SE(3) Lie group basics, exp/log, Jacobians.

- **Joan Solà "A micro Lie theory"** (free PDF, arXiv 1812.01537)
  - §6: right Jacobian `J_r`, left Jacobian, inverse.
  - §7: how to differentiate through SO(3)/SE(3) with the chain rule.
    Essential for understanding where `J_r^{-1}(r_R)` comes from.

---

## Part 1 — The cost function

Pure visual BA minimises reprojection error over poses and map points:

```
E_visual = ½ Σ_{obs} ‖ π(T_i, X_j) − z_ij ‖²_Σ_cam
```

VI-BA adds an **IMU factor** for every consecutive keyframe pair:

```
E = E_visual + ½ Σ_{edges i→j} r_imu(i,j)ᵀ · Ω_ij · r_imu(i,j)
```

`Ω_ij = Σ_ij⁻¹` is the 9×9 inverse of the preintegration covariance
(`PreintegratedImu::covariance` in `imu.rs:73`).

Each keyframe now has a **15-DOF state**:

```
x_i = ( R_i ∈ SO(3),  p_i ∈ R³,  v_i ∈ R³,  b_g_i ∈ R³,  b_a_i ∈ R³ )
       └─────── 6-DOF pose ────────┘   vel     gyro bias   accel bias
```

The tangent vector solved by the optimiser per free keyframe:

```
δx_i = [ ρ_i(3),  ω_i(3),  δv_i(3),  δb_g_i(3),  δb_a_i(3) ]
         └── pose tangent ──┘
         (same [ρ; ω] convention as the existing reprojection code)
```

The velocity and bias components are updated additively; the pose component uses
the existing `SE3F32::retract` call.

---

## Part 2 — IMU preintegration recap

`PreintegratedImu::predict` (`imu.rs:269`) encodes the constraint between
consecutive keyframe states:

```
R_j   ≈  R_i · ΔR_ij
v_j   ≈  v_i + g·Δt  +  R_i · Δv_ij
p_j   ≈  p_i + v_i·Δt  +  ½g·Δt²  +  R_i · Δp_ij
```

**Convention:** `R_i` here is the **body-to-world** rotation.
In the `ViBaKeyframe` struct, `pose.rotation` is the *world-to-camera* matrix.
So:

```rust
// Body-to-world rotation (R_wb_i):
let r_wb_i = kf_i.pose.rotation.transpose();

// Body/camera centre in world frame (p_i):
let p_i = -(r_wb_i * kf_i.pose.translation);

// Velocity is already in world frame:
let v_i = kf_i.velocity;
```

The **bias-corrected** deltas are computed via the `_with_bias` methods
(`imu.rs:110–128`), which apply the first-order correction stored in the bias
Jacobians without re-integrating raw measurements:

```rust
let dr = preint.delta_rotation_with_bias(&kf_i.bias); // ΔR_ij ∈ SO(3)
let dv = preint.delta_velocity_with_bias(&kf_i.bias); // Δv_ij ∈ R³
let dp = preint.delta_position_with_bias(&kf_i.bias); // Δp_ij ∈ R³
```

---

## Part 3 — The 9-DOF IMU residual

The residual is the difference between what the preintegrated IMU *predicts*
and what the current keyframe states actually give (Forster eq. 21–23):

```
r_R = Log( ΔR_ij^T · R_wb_i^T · R_wb_j )         ← 3-DOF rotation residual
r_v = R_wb_i^T · (v_j − v_i − g·Δt)  − Δv_ij     ← 3-DOF velocity residual
r_p = R_wb_i^T · (p_j − p_i − v_i·Δt − ½g·Δt²) − Δp_ij  ← 3-DOF position residual
```

`Log` is the SO(3) logarithm, which returns a 3-vector (the axis-angle /
rotation vector).  At the optimum all three blocks are zero.

In code:

```rust
// Note: R_wb_i^T = R_wc_i = kf_i.pose.rotation
let r_wc_i = kf_i.pose.rotation; // world-to-cam = body^T-to-world^T = R_wb_i^T

let r_R_mat = dr.transpose() * (r_wc_i * r_wb_j); // ΔR^T · R_wb_i^T · R_wb_j
let r_R = SO3F64::log(r_R_mat); // 3-vec

let b_vec = v_j - v_i - params.gravity * dt;
let r_v   = r_wc_i * b_vec - dv;               // R_wb_i^T · b − Δv

let a_vec = p_j - p_i - v_i * dt - params.gravity * (0.5 * dt * dt);
let r_p   = r_wc_i * a_vec - dp;               // R_wb_i^T · a − Δp
```

---

## Part 4 — Jacobians

> **Derivations:** Forster et al. TRO 2017 Table I gives all of these.
> ORB-SLAM3 Appendix A re-derives them in the world-frame convention.
> Joan Solà §6 explains the right Jacobian and its inverse.

### Notation

- `R_i = R_wb_i`, `R_j = R_wb_j` (body-to-world, = `pose.rotation.transpose()`)
- `a = p_j − p_i − v_i·Δt − ½g·Δt²` (argument of `r_p`)
- `b = v_j − v_i − g·Δt` (argument of `r_v`)
- `Jr_inv` = inverse right Jacobian of SO(3) evaluated at `r_R` (see §6 of Solà)

### 4.1 Rotation residual `r_R` (3-DOF)

| ∂r_R / ∂ | Block (3×3) |
|---|---|
| `ω_i` (rotation of kf i) | `+Jr_inv · R_j^T` |
| `ω_j` (rotation of kf j) | `−Jr_inv · R_j^T` |
| `b_g_i` (gyro bias) | `−Jr_inv · ΔR^T · d_rotation_d_bias_gyro` |

> **Sign note for `ω_i` vs `ω_j`:** The existing BA stores world-to-camera
> poses and perturbs them with right-multiplication `R_wc → R_wc · Exp(ω)`.
> This induces a **left** perturbation on `R_wb = R_wc^T`:
> `R_wb_new ≈ (I − [ω]_×) · R_wb`.
> Forster et al. use **right** perturbation on `R_wb`.
> The rotation Jacobians above are written for the left-perturbation convention
> (matching the existing code).  If you see the residual going the wrong way
> under a small step, flip the sign.

### 4.2 Velocity residual `r_v` (3-DOF)

| ∂r_v / ∂ | Block (3×3) |
|---|---|
| `ω_i` | `[R_wb_i^T · b]_×`  (skew of `r_v + dv`) |
| `v_i` | `−R_wb_i^T` = `−R_wc_i` = `−pose_i.rotation` |
| `v_j` | `+R_wb_i^T` = `+pose_i.rotation` |
| `b_g_i` | `−d_velocity_d_bias_gyro` |
| `b_a_i` | `−d_velocity_d_bias_accel` |

### 4.3 Position residual `r_p` (3-DOF)

| ∂r_p / ∂ | Block (3×3) |
|---|---|
| `ω_i` | `[R_wb_i^T · a]_×`  (skew of `r_p + dp`) |
| `ρ_i` (translation of kf i) | `+R_wb_i^T` = `+pose_i.rotation` |
| `ρ_j` (translation of kf j) | `−R_wb_i^T` = `−pose_i.rotation` |
| `v_i` | `−pose_i.rotation · Δt` |
| `b_g_i` | `−d_position_d_bias_gyro` |
| `b_a_i` | `−d_position_d_bias_accel` |

### 4.4 Assembling the 9×15 Jacobian

Each IMU edge contributes a **9×15 block at keyframe i** and a **9×15 block at
keyframe j**.  Layout of the 9 residual rows: `[r_R(0:3), r_v(3:6), r_p(6:9)]`.
Layout of the 15 state columns: `[ρ(0:3), ω(3:6), v(6:9), b_g(9:12), b_a(12:15)]`.

Jacobian at keyframe i (`J_i`, 9×15):

```
       ρ_i   ω_i    v_i   b_g_i  b_a_i
r_R  [  0   Jr_inv·Rj^T   0   −Jr_inv·ΔR^T·dR_dbg   0  ]
r_v  [  0   [Ri^T·b]_×  −Ri^T       −dvdbg          −dvdba ]
r_p  [ Ri^T [Ri^T·a]_×  −Ri^T·Δt   −dpdbg          −dpdba ]
```

Jacobian at keyframe j (`J_j`, 9×15):

```
       ρ_j   ω_j    v_j   b_g_j  b_a_j
r_R  [  0  −Jr_inv·Rj^T  0    0    0  ]
r_v  [  0       0      +Ri^T  0    0  ]
r_p  [ −Ri^T    0       0     0    0  ]
```

The weighted normal-equation contribution:

```
A_kf[i,i] += J_i^T · Ω · J_i      (15×15 block on diagonal)
A_kf[j,j] += J_j^T · Ω · J_j      (15×15 block on diagonal)
A_kf[i,j] += J_i^T · Ω · J_j      (15×15 off-diagonal block)
A_kf[j,i] += J_j^T · Ω · J_i      (symmetric)
g_kf[i]   -= J_i^T · Ω · r_imu    (15-vector)
g_kf[j]   -= J_j^T · Ω · r_imu    (15-vector)
```

---

## Part 5 — Hessian structure and Schur complement

### 5.1 Block structure

```
H = [ A_kf   B   ]
    [ B^T    C   ]
```

| Block | Size | Meaning |
|---|---|---|
| `A_kf` | 15P × 15P | Keyframe (pose+vel+bias) Hessian; **NOT** block-diagonal — IMU edges couple adjacent frames off-diagonally |
| `C` | 3N × 3N | Map point Hessian; **still block-diagonal** (each 3×3) |
| `B` | 15P × 3N | Cross-terms; visual only (IMU never touches map points) — the 6×3 pose-point block sits in the first 6 rows of each 15-row stripe |

### 5.2 Schur complement (unchanged math)

Because C is still block-diagonal, the Schur elimination is the same as the
existing visual BA:

```
M = A_kf − B · C⁻¹ · B^T       (dense 15P × 15P, solve with Cholesky)
m = g_kf − B · C⁻¹ · g_point
```

After Cholesky: `M · δ_kf = m` → 15P-vector `δ_kf`.
Back-substitute points: `δ_x[j] = C⁻¹ · (g_pt[j] − B^T[j] · δ_kf[pose_part])`.

The existing `b_by_point` / `c_inv_blocks` / Schur-fill code in `ba_schur.rs`
is reused with only one change: `A_kf` is now `15P × 15P` instead of `6P × 6P`.

---

## Part 6 — Code scaffold

```rust
/// Visual-inertial bundle adjustment via dense Schur-complement reduction.
///
/// State per keyframe: [ρ(3), ω(3), v(3), b_g(3), b_a(3)] = 15-DOF.
/// Visual observations contribute to the pose (first 6) columns only.
/// IMU edges contribute 15×15 blocks to A_kf for both endpoints.
/// Map points are eliminated via the Schur complement (C is block-diagonal).
pub fn bundle_adjust_vi(
    keyframes: &[ViBaKeyframe],
    points: &[Vec3F64],
    observations: &[BaObservation],
    imu_edges: &[ImuEdge],
    camera: &PinholeCamera,
    params: &ViBaParams,
) -> Result<ViBaResult, ViBaError> {
    const KF_DIM: usize = 15; // DOF per free keyframe

    let nkf = keyframes.len();
    let npt = points.len();

    // ── Free/fixed index maps ─────────────────────────────────────────────
    let kf_local: Vec<i64> = {
        let mut v = vec![-1i64; nkf];
        let mut next = 0i64;
        for i in 0..nkf {
            if !keyframes[i].fixed { v[i] = next; next += 1; }
        }
        v
    };
    let pt_local: Vec<i64> = { /* same pattern */ vec![-1i64; npt] };
    let n_free_kf = kf_local.iter().filter(|&&x| x >= 0).count();
    let n_free_pt = pt_local.iter().filter(|&&x| x >= 0).count();

    if n_free_kf == 0 { return Err(ViBaError::NoFreeVariables); }

    let mut kf_state: Vec<ViBaKeyframe> = keyframes.to_vec();
    let mut xyz: Vec<Vec3F64> = points.to_vec();
    let mut lambda = params.initial_lambda;
    let mut converged = false;
    let mut iters_done = 0usize;

    for _iter in 0..params.max_iterations {
        iters_done += 1;

        // ── Allocate normal-equation blocks ───────────────────────────────
        let dim = n_free_kf * KF_DIM;
        let mut a_mat = vec![0.0_f64; dim * dim]; // row-major 15P×15P
        let mut g_kf  = vec![0.0_f64; dim];
        let mut c_blocks = vec![[0.0_f32; 9]; n_free_pt];
        let mut g_point  = vec![[0.0_f32; 3]; n_free_pt];
        let mut b_by_point: Vec<Vec<(usize, [f32; 18])>> = vec![vec![]; n_free_pt];
        let mut cost = 0.0_f64;

        // ── Visual observations ───────────────────────────────────────────
        // The existing residual_and_jacobians returns (r 2-vec, J_pose 2×6, J_point 2×3).
        // J_pose goes into the top-left 6×6 block of A_kf[kli]; C, B, g_point unchanged.
        for obs in observations {
            let kli = kf_local[obs.pose_idx];
            let xli = pt_local[obs.point_idx];
            let se3 = pose_to_se3(&kf_state[obs.pose_idx].pose);
            let (r, j_pose, j_point) =
                residual_and_jacobians(&se3, &xyz[obs.point_idx], obs.pixel, camera);
            cost += 0.5 * (r[0] * r[0] + r[1] * r[1]) as f64;

            if kli >= 0 {
                // Accumulate J_pose^T J_pose into the 6×6 top-left corner of
                // a_mat block (kli*KF_DIM .. kli*KF_DIM+6, same cols).
                let base = kli as usize * KF_DIM;
                for i in 0..6 { for k in 0..6 {
                    a_mat[(base+i)*dim + base+k] +=
                        (j_pose[i]*j_pose[k] + j_pose[6+i]*j_pose[6+k]) as f64;
                }}
                for i in 0..6 {
                    g_kf[base + i] -=
                        (j_pose[i]*r[0] + j_pose[6+i]*r[1]) as f64;
                }
            }
            // C, g_point, b_by_point: identical to existing bundle_adjust_schur
            // (the point-block cross-term only involves the first 6 cols of kf state)
            // ... (copy from existing code) ...
        }

        // ── IMU edges ─────────────────────────────────────────────────────
        for edge in imu_edges {
            let kli = kf_local[edge.from_idx];
            let klj = kf_local[edge.to_idx];
            // Skip if both endpoints are fixed.
            if kli < 0 && klj < 0 { continue; }

            let kf_i = &kf_state[edge.from_idx];
            let kf_j = &kf_state[edge.to_idx];
            let pre   = &edge.preintegrated;
            let dt    = pre.dt;
            let g     = params.gravity;

            // State quantities
            let r_wb_i = kf_i.pose.rotation.transpose(); // body-to-world at i
            let r_wb_j = kf_j.pose.rotation.transpose();
            let r_wc_i = kf_i.pose.rotation;             // = R_wb_i^T
            let p_i    = -(r_wb_i * kf_i.pose.translation);
            let p_j    = -(r_wb_j * kf_j.pose.translation);
            let v_i    = kf_i.velocity;
            let v_j    = kf_j.velocity;

            // Bias-corrected deltas
            let dr = pre.delta_rotation_with_bias(&kf_i.bias);
            let dv = pre.delta_velocity_with_bias(&kf_i.bias);
            let dp = pre.delta_position_with_bias(&kf_i.bias);

            // Residuals
            // r_R: 3-vec
            let r_r_mat = dr.transpose() * (r_wc_i * r_wb_j);
            let r_r = so3_log(r_r_mat); // implement SO3 log
            // r_v: 3-vec
            let b_vec = v_j - v_i - g * dt;
            let r_v   = r_wc_i * b_vec - dv;
            // r_p: 3-vec
            let a_vec = p_j - p_i - v_i * dt - g * (0.5 * dt * dt);
            let r_p   = r_wc_i * a_vec - dp;

            // Stack residual: r_imu = [r_R(3), r_v(3), r_p(3)]
            let r_imu: [f64; 9] = [
                r_r.x, r_r.y, r_r.z,
                r_v.x, r_v.y, r_v.z,
                r_p.x, r_p.y, r_p.z,
            ];
            cost += 0.5 * r_imu.iter().map(|x| x*x).sum::<f64>(); // unweighted cost; use Ω below

            // Information matrix Ω = covariance^{-1} (9×9, column-major)
            let omega = invert_9x9(&pre.covariance); // you need to implement this

            // Right-Jacobian inverse at r_R
            let jr_inv = so3_right_jacobian_inverse(r_r); // 3×3

            // Bias Jacobians from preintegration
            let dr_dbg = pre.d_rotation_d_bias_gyro;  // 3×3
            let dv_dbg = pre.d_velocity_d_bias_gyro;  // 3×3
            let dv_dba = pre.d_velocity_d_bias_accel; // 3×3
            let dp_dbg = pre.d_position_d_bias_gyro;  // 3×3
            let dp_dba = pre.d_position_d_bias_accel; // 3×3

            // Assemble J_i (9×15) and J_j (9×15).
            // Column layout: [ρ(0:3), ω(3:6), v(6:9), b_g(9:12), b_a(12:15)]
            // Row layout:    [r_R(0:3), r_v(3:6), r_p(6:9)]
            //
            // J_i:
            //   r_R row: ρ→0, ω→ Jr_inv·Rj_wc^T (=Jr_inv·Rwb_j), v→0, bg→−Jr_inv·ΔR^T·dr_dbg, ba→0
            //   r_v row: ρ→0, ω→skew(r_v+dv),    v→−R_wc_i,       bg→−dv_dbg, ba→−dv_dba
            //   r_p row: ρ→R_wc_i, ω→skew(r_p+dp), v→−R_wc_i·dt,  bg→−dp_dbg, ba→−dp_dba
            //
            // J_j:
            //   r_R row: ρ→0, ω→−Jr_inv·Rwb_j,  rest→0
            //   r_v row: ρ→0, ω→0, v→+R_wc_i, rest→0
            //   r_p row: ρ→−R_wc_i, rest→0

            // Build J_i as flat 9×15 row-major (or column-major — be consistent)
            // then compute:
            //   A_kf[i,i] += J_i^T Ω J_i      g_kf[i] -= J_i^T Ω r_imu
            //   A_kf[j,j] += J_j^T Ω J_j      g_kf[j] -= J_j^T Ω r_imu
            //   A_kf[i,j] += J_i^T Ω J_j      A_kf[j,i] = A_kf[i,j]^T

            // ... (implement the accumulation) ...
            let _ = (kli, klj, jr_inv, omega, dr, dr_dbg, dv_dbg, dv_dba, dp_dbg, dp_dba);
        }

        // ── LM damping ────────────────────────────────────────────────────
        for k in 0..n_free_kf * KF_DIM {
            a_mat[k * dim + k] += lambda;
        }
        for cb in &mut c_blocks {
            for d in 0..3 { cb[d*3+d] += lambda as f32; }
        }

        // ── Schur eliminate points: M = A_kf − B·C⁻¹·B^T ────────────────
        // (identical logic to existing bundle_adjust_schur_with_priors,
        //  but operating on dim=15P instead of 6P)
        let mut m_mat = Mat::<f64>::zeros(dim, dim);
        let mut m_vec = vec![0.0_f64; dim];
        // ... copy Schur fill from existing code ...

        // ── Cholesky solve ────────────────────────────────────────────────
        // symmetrise M first (same as existing code)
        let chol = m_mat.llt(faer::Side::Lower)
            .map_err(|e| ViBaError::CholeskyFailed(format!("{e:?}")))?;
        let rhs = Mat::<f64>::from_fn(dim, 1, |i, _| m_vec[i]);
        let d_kf_col = chol.solve(&rhs);

        // ── Back-substitute points ────────────────────────────────────────
        // (identical to existing code; δ_pose comes from first 6 elements of each
        //  15-block in d_kf_col; point back-substitution uses B^T · δ_pose)

        // ── Retract ───────────────────────────────────────────────────────
        for i_global in 0..nkf {
            let kli = kf_local[i_global];
            if kli < 0 { continue; }
            let base = kli as usize * KF_DIM;
            let delta_pose: [f32; 6] = std::array::from_fn(|k| d_kf_col[(base+k, 0)] as f32);
            let se3 = pose_to_se3(&kf_state[i_global].pose);
            kf_state[i_global].pose = se3_to_pose(&se3.retract(&delta_pose));
            kf_state[i_global].velocity.x += d_kf_col[(base+6, 0)];
            kf_state[i_global].velocity.y += d_kf_col[(base+7, 0)];
            kf_state[i_global].velocity.z += d_kf_col[(base+8, 0)];
            kf_state[i_global].bias.gyro.x += d_kf_col[(base+9,  0)];
            kf_state[i_global].bias.gyro.y += d_kf_col[(base+10, 0)];
            kf_state[i_global].bias.gyro.z += d_kf_col[(base+11, 0)];
            kf_state[i_global].bias.accel.x += d_kf_col[(base+12, 0)];
            kf_state[i_global].bias.accel.y += d_kf_col[(base+13, 0)];
            kf_state[i_global].bias.accel.z += d_kf_col[(base+14, 0)];
        }

        // ... LM accept/reject cost comparison, lambda update, convergence check ...
    }

    Ok(ViBaResult {
        keyframes: kf_state,
        points: xyz,
        iterations: iters_done,
        converged,
        final_cost: 0.0,
    })
}
```

---

## Part 7 — Utility functions needed

### 7.1 `so3_log` — SO(3) logarithm

Inverse of `SO3F64::exp`.  Returns the rotation vector `ω` such that `Exp(ω) = R`.

```rust
fn so3_log(r: Mat3F64) -> Vec3F64 {
    let trace = r.col(0).x + r.col(1).y + r.col(2).z;
    let cos_angle = ((trace - 1.0) * 0.5).clamp(-1.0, 1.0);
    let angle = cos_angle.acos();
    if angle.abs() < 1e-8 {
        // Small angle: use first-order approximation
        return Vec3F64::new(
            r.col(1).z - r.col(2).y,
            r.col(2).x - r.col(0).z,
            r.col(0).y - r.col(1).x,
        ) * 0.5;
    }
    let s = angle / (2.0 * angle.sin());
    Vec3F64::new(
        (r.col(1).z - r.col(2).y) * s,
        (r.col(2).x - r.col(0).z) * s,
        (r.col(0).y - r.col(1).x) * s,
    )
}
```

Reference: slambook-en Ch. 4, eq. 4.22; Solà §6.2.

### 7.2 `so3_right_jacobian_inverse` — `Jr^{-1}(ω)`

Used in the rotation residual Jacobians.

```rust
fn so3_right_jacobian_inverse(omega: Vec3F64) -> Mat3F64 {
    let theta = (omega.x*omega.x + omega.y*omega.y + omega.z*omega.z).sqrt();
    let skew = skew3(omega);
    let skew2 = skew * skew; // skew · skew (Mat3×Mat3)
    if theta < 1e-5 {
        // First-order: Jr^{-1} ≈ I + ½[ω]_×
        return Mat3F64::IDENTITY + skew * 0.5;
    }
    let c = (1.0 / (theta * theta))
          - (1.0 + theta.cos()) / (2.0 * theta * theta.sin());
    Mat3F64::IDENTITY + skew * 0.5 + skew2 * c
}
```

Reference: Forster eq. A.8; Solà §6, eq. (99).

### 7.3 `skew3` — skew-symmetric matrix

```rust
fn skew3(v: Vec3F64) -> Mat3F64 {
    Mat3F64::from_cols(
        Vec3F64::new(  0.0,  v.z, -v.y),
        Vec3F64::new( -v.z,  0.0,  v.x),
        Vec3F64::new(  v.y, -v.x,  0.0),
    )
}
```

### 7.4 `invert_9x9` — 9×9 matrix inverse

Use `faer` directly rather than hand-rolling:

```rust
fn invert_9x9(m: &[f64; 81]) -> [f64; 81] {
    let mat = Mat::<f64>::from_fn(9, 9, |r, c| m[r + c * 9]); // column-major input
    let inv = mat.partial_piv_lu().inverse();
    let mut out = [0.0_f64; 81];
    for r in 0..9 { for c in 0..9 { out[r + c * 9] = inv[(r, c)]; } }
    out
}
```

---

## Part 8 — Testing strategy (strongly recommended)

### Step 1: Test `so3_log` against `so3_exp`

```
assert ‖Log(Exp(ω)) − ω‖ < ε   for random ω with ‖ω‖ < π
```

### Step 2: Test IMU residual against `predict`

Given ground-truth consecutive keyframe states (R_i, p_i, v_i) and a
`PreintegratedImu` that correctly integrates the motion between them:
```
r_imu should be ≈ 0 at the true states
```

### Step 3: Finite-difference Jacobian check

For each column `k` of `J_i`:
```
(r_imu(x_i + ε·e_k) − r_imu(x_i)) / ε  ≈  J_i[:, k]
```
This is the most reliable way to catch sign errors in the rotation Jacobians.

### Step 4: Two-keyframe synthetic test

Create two keyframes with known relative motion, add one IMU edge with a
known constant-acceleration measurement.  VI-BA should:
- Recover velocity at both keyframes
- Converge in a small number of iterations

---

## Summary checklist

| Step | What to implement | Where to look |
|---|---|---|
| 1 | `so3_log`, `skew3` | This doc §7; slambook Ch. 4 |
| 2 | `so3_right_jacobian_inverse` | Solà §6; Forster App. A |
| 3 | `invert_9x9` via faer | §7.4 |
| 4 | `imu_residual_and_jacobians()` | §3–4; Forster Table I |
| 5 | IMU block accumulation into `a_mat` | §4.4; §6 scaffold |
| 6 | Extend Schur/Cholesky to 15P dim | §5; existing code |
| 7 | Retract all 15 DOF per keyframe | §6 scaffold |
| 8 | Wire up from `pipeline.rs` | pipeline.rs line 843 |


# Visual-Inertial Bundle Adjustment: Math to Code Guide

This document covers the mathematics behind VI-BA and maps every equation to
the corresponding code in this repository.  Read it alongside the two reference
books already in the repo root (`slambook-en.pdf` and `SLAM Handbook.pdf`).

---

## Reference map

| Topic | Primary source | Where in this repo |
|---|---|---|
| IMU preintegration on manifold | Forster et al. TRO 2017 §III–IV | `crates/kornia-sensors/src/imu.rs` |
| VI-BA residuals and Jacobians | Forster et al. TRO 2017 Table I; ORB-SLAM3 Appendix A | `crates/kornia-slam/src/ba_schur.rs` |
| Schur-complement BA | Triggs et al. 1999 §5; slambook-en Ch. 9 | `ba_schur.rs` (existing visual BA) |
| SO(3) / SE(3) Lie theory | Joan Solà "A micro Lie theory" (arXiv 1812.01537) §6–7 | `kornia-algebra` |
| Covariance propagation | Forster et al. TRO 2017 §IV-B | `imu.rs::integrate` |

### Where to read the derivations

- **Forster et al. TRO 2017** — "On-Manifold Preintegration for Real-Time
  Visual–Inertial Odometry".  This is the canonical source for the
  preintegration model, the 9-DOF residual, and the bias Jacobians.
  - §III: manifold preintegration (what `PreintegratedImu::integrate` does)
  - §IV-A: residuals `r_R, r_v, r_p` (equations 21–23)
  - Table I: all analytical Jacobians used in the normal equations
  - §IV-B: covariance propagation (`A`, `B`, `N` matrices in `imu.rs:178–215`)

- **ORB-SLAM3 paper** — Campos et al. TPAMI 2021.
  - Appendix A: step-by-step Jacobian derivation in the world-frame
    right-perturbation convention (the cleanest reference for implementation).
  - §IV: how VI-BA fits into the full system (local window, marginalisation).

- **SLAM Handbook.pdf** (in repo root)
  - Part II covers factor-graph BA, information matrices, and the Schur trick
    in a unified notation.

- **slambook-en.pdf** (in repo root)
  - Chapter 9: Schur complement, marginalisation, bundle adjustment.
  - Chapter 4: SO(3) / SE(3) Lie group basics, exp/log, Jacobians.

- **Joan Solà "A micro Lie theory"** (free PDF, arXiv 1812.01537)
  - §6: right Jacobian `J_r`, left Jacobian, inverse.
  - §7: how to differentiate through SO(3)/SE(3) with the chain rule.
    Essential for understanding where `J_r^{-1}(r_R)` comes from.

---

## Part 1 — The cost function

Pure visual BA minimises reprojection error over poses and map points:

```
E_visual = ½ Σ_{obs} ‖ π(T_i, X_j) − z_ij ‖²_Σ_cam
```

VI-BA adds an **IMU factor** for every consecutive keyframe pair:

```
E = E_visual + ½ Σ_{edges i→j} r_imu(i,j)ᵀ · Ω_ij · r_imu(i,j)
```

`Ω_ij = Σ_ij⁻¹` is the 9×9 inverse of the preintegration covariance
(`PreintegratedImu::covariance` in `imu.rs:73`).

Each keyframe now has a **15-DOF state**:

```
x_i = ( R_i ∈ SO(3),  p_i ∈ R³,  v_i ∈ R³,  b_g_i ∈ R³,  b_a_i ∈ R³ )
       └─────── 6-DOF pose ────────┘   vel     gyro bias   accel bias
```

The tangent vector solved by the optimiser per free keyframe:

```
δx_i = [ ρ_i(3),  ω_i(3),  δv_i(3),  δb_g_i(3),  δb_a_i(3) ]
         └── pose tangent ──┘
         (same [ρ; ω] convention as the existing reprojection code)
```

The velocity and bias components are updated additively; the pose component uses
the existing `SE3F32::retract` call.

---

## Part 2 — IMU preintegration recap

`PreintegratedImu::predict` (`imu.rs:269`) encodes the constraint between
consecutive keyframe states:

```
R_j   ≈  R_i · ΔR_ij
v_j   ≈  v_i + g·Δt  +  R_i · Δv_ij
p_j   ≈  p_i + v_i·Δt  +  ½g·Δt²  +  R_i · Δp_ij
```

**Convention:** `R_i` here is the **body-to-world** rotation.
In the `ViBaKeyframe` struct, `pose.rotation` is the *world-to-camera* matrix.
So:

```rust
// Body-to-world rotation (R_wb_i):
let r_wb_i = kf_i.pose.rotation.transpose();

// Body/camera centre in world frame (p_i):
let p_i = -(r_wb_i * kf_i.pose.translation);

// Velocity is already in world frame:
let v_i = kf_i.velocity;
```

The **bias-corrected** deltas are computed via the `_with_bias` methods
(`imu.rs:110–128`), which apply the first-order correction stored in the bias
Jacobians without re-integrating raw measurements:

```rust
let dr = preint.delta_rotation_with_bias(&kf_i.bias); // ΔR_ij ∈ SO(3)
let dv = preint.delta_velocity_with_bias(&kf_i.bias); // Δv_ij ∈ R³
let dp = preint.delta_position_with_bias(&kf_i.bias); // Δp_ij ∈ R³
```

---

## Part 3 — The 9-DOF IMU residual

The residual is the difference between what the preintegrated IMU *predicts*
and what the current keyframe states actually give (Forster eq. 21–23):

```
r_R = Log( ΔR_ij^T · R_wb_i^T · R_wb_j )         ← 3-DOF rotation residual
r_v = R_wb_i^T · (v_j − v_i − g·Δt)  − Δv_ij     ← 3-DOF velocity residual
r_p = R_wb_i^T · (p_j − p_i − v_i·Δt − ½g·Δt²) − Δp_ij  ← 3-DOF position residual
```

`Log` is the SO(3) logarithm, which returns a 3-vector (the axis-angle /
rotation vector).  At the optimum all three blocks are zero.

In code:

```rust
// Note: R_wb_i^T = R_wc_i = kf_i.pose.rotation
let r_wc_i = kf_i.pose.rotation; // world-to-cam = body^T-to-world^T = R_wb_i^T

let r_R_mat = dr.transpose() * (r_wc_i * r_wb_j); // ΔR^T · R_wb_i^T · R_wb_j
let r_R = SO3F64::log(r_R_mat); // 3-vec

let b_vec = v_j - v_i - params.gravity * dt;
let r_v   = r_wc_i * b_vec - dv;               // R_wb_i^T · b − Δv

let a_vec = p_j - p_i - v_i * dt - params.gravity * (0.5 * dt * dt);
let r_p   = r_wc_i * a_vec - dp;               // R_wb_i^T · a − Δp
```

---

## Part 4 — Jacobians

> **Derivations:** Forster et al. TRO 2017 Table I gives all of these.
> ORB-SLAM3 Appendix A re-derives them in the world-frame convention.
> Joan Solà §6 explains the right Jacobian and its inverse.

### Notation

- `R_i = R_wb_i`, `R_j = R_wb_j` (body-to-world, = `pose.rotation.transpose()`)
- `a = p_j − p_i − v_i·Δt − ½g·Δt²` (argument of `r_p`)
- `b = v_j − v_i − g·Δt` (argument of `r_v`)
- `Jr_inv` = inverse right Jacobian of SO(3) evaluated at `r_R` (see §6 of Solà)

### 4.1 Rotation residual `r_R` (3-DOF)

| ∂r_R / ∂ | Block (3×3) |
|---|---|
| `ω_i` (rotation of kf i) | `+Jr_inv · R_j^T` |
| `ω_j` (rotation of kf j) | `−Jr_inv · R_j^T` |
| `b_g_i` (gyro bias) | `−Jr_inv · ΔR^T · d_rotation_d_bias_gyro` |

> **Sign note for `ω_i` vs `ω_j`:** The existing BA stores world-to-camera
> poses and perturbs them with right-multiplication `R_wc → R_wc · Exp(ω)`.
> This induces a **left** perturbation on `R_wb = R_wc^T`:
> `R_wb_new ≈ (I − [ω]_×) · R_wb`.
> Forster et al. use **right** perturbation on `R_wb`.
> The rotation Jacobians above are written for the left-perturbation convention
> (matching the existing code).  If you see the residual going the wrong way
> under a small step, flip the sign.

### 4.2 Velocity residual `r_v` (3-DOF)

| ∂r_v / ∂ | Block (3×3) |
|---|---|
| `ω_i` | `[R_wb_i^T · b]_×`  (skew of `r_v + dv`) |
| `v_i` | `−R_wb_i^T` = `−R_wc_i` = `−pose_i.rotation` |
| `v_j` | `+R_wb_i^T` = `+pose_i.rotation` |
| `b_g_i` | `−d_velocity_d_bias_gyro` |
| `b_a_i` | `−d_velocity_d_bias_accel` |

### 4.3 Position residual `r_p` (3-DOF)

| ∂r_p / ∂ | Block (3×3) |
|---|---|
| `ω_i` | `[R_wb_i^T · a]_×`  (skew of `r_p + dp`) |
| `ρ_i` (translation of kf i) | `+R_wb_i^T` = `+pose_i.rotation` |
| `ρ_j` (translation of kf j) | `−R_wb_i^T` = `−pose_i.rotation` |
| `v_i` | `−pose_i.rotation · Δt` |
| `b_g_i` | `−d_position_d_bias_gyro` |
| `b_a_i` | `−d_position_d_bias_accel` |

### 4.4 Assembling the 9×15 Jacobian

Each IMU edge contributes a **9×15 block at keyframe i** and a **9×15 block at
keyframe j**.  Layout of the 9 residual rows: `[r_R(0:3), r_v(3:6), r_p(6:9)]`.
Layout of the 15 state columns: `[ρ(0:3), ω(3:6), v(6:9), b_g(9:12), b_a(12:15)]`.

Jacobian at keyframe i (`J_i`, 9×15):

```
       ρ_i   ω_i    v_i   b_g_i  b_a_i
r_R  [  0   Jr_inv·Rj^T   0   −Jr_inv·ΔR^T·dR_dbg   0  ]
r_v  [  0   [Ri^T·b]_×  −Ri^T       −dvdbg          −dvdba ]
r_p  [ Ri^T [Ri^T·a]_×  −Ri^T·Δt   −dpdbg          −dpdba ]
```

Jacobian at keyframe j (`J_j`, 9×15):

```
       ρ_j   ω_j    v_j   b_g_j  b_a_j
r_R  [  0  −Jr_inv·Rj^T  0    0    0  ]
r_v  [  0       0      +Ri^T  0    0  ]
r_p  [ −Ri^T    0       0     0    0  ]
```

The weighted normal-equation contribution:

```
A_kf[i,i] += J_i^T · Ω · J_i      (15×15 block on diagonal)
A_kf[j,j] += J_j^T · Ω · J_j      (15×15 block on diagonal)
A_kf[i,j] += J_i^T · Ω · J_j      (15×15 off-diagonal block)
A_kf[j,i] += J_j^T · Ω · J_i      (symmetric)
g_kf[i]   -= J_i^T · Ω · r_imu    (15-vector)
g_kf[j]   -= J_j^T · Ω · r_imu    (15-vector)
```

---

## Part 5 — Hessian structure and Schur complement

### 5.1 Block structure

```
H = [ A_kf   B   ]
    [ B^T    C   ]
```

| Block | Size | Meaning |
|---|---|---|
| `A_kf` | 15P × 15P | Keyframe (pose+vel+bias) Hessian; **NOT** block-diagonal — IMU edges couple adjacent frames off-diagonally |
| `C` | 3N × 3N | Map point Hessian; **still block-diagonal** (each 3×3) |
| `B` | 15P × 3N | Cross-terms; visual only (IMU never touches map points) — the 6×3 pose-point block sits in the first 6 rows of each 15-row stripe |

### 5.2 Schur complement (unchanged math)

Because C is still block-diagonal, the Schur elimination is the same as the
existing visual BA:

```
M = A_kf − B · C⁻¹ · B^T       (dense 15P × 15P, solve with Cholesky)
m = g_kf − B · C⁻¹ · g_point
```

After Cholesky: `M · δ_kf = m` → 15P-vector `δ_kf`.
Back-substitute points: `δ_x[j] = C⁻¹ · (g_pt[j] − B^T[j] · δ_kf[pose_part])`.

The existing `b_by_point` / `c_inv_blocks` / Schur-fill code in `ba_schur.rs`
is reused with only one change: `A_kf` is now `15P × 15P` instead of `6P × 6P`.

---

## Part 6 — Code scaffold

```rust
/// Visual-inertial bundle adjustment via dense Schur-complement reduction.
///
/// State per keyframe: [ρ(3), ω(3), v(3), b_g(3), b_a(3)] = 15-DOF.
/// Visual observations contribute to the pose (first 6) columns only.
/// IMU edges contribute 15×15 blocks to A_kf for both endpoints.
/// Map points are eliminated via the Schur complement (C is block-diagonal).
pub fn bundle_adjust_vi(
    keyframes: &[ViBaKeyframe],
    points: &[Vec3F64],
    observations: &[BaObservation],
    imu_edges: &[ImuEdge],
    camera: &PinholeCamera,
    params: &ViBaParams,
) -> Result<ViBaResult, ViBaError> {
    const KF_DIM: usize = 15; // DOF per free keyframe

    let nkf = keyframes.len();
    let npt = points.len();

    // ── Free/fixed index maps ─────────────────────────────────────────────
    let kf_local: Vec<i64> = {
        let mut v = vec![-1i64; nkf];
        let mut next = 0i64;
        for i in 0..nkf {
            if !keyframes[i].fixed { v[i] = next; next += 1; }
        }
        v
    };
    let pt_local: Vec<i64> = { /* same pattern */ vec![-1i64; npt] };
    let n_free_kf = kf_local.iter().filter(|&&x| x >= 0).count();
    let n_free_pt = pt_local.iter().filter(|&&x| x >= 0).count();

    if n_free_kf == 0 { return Err(ViBaError::NoFreeVariables); }

    let mut kf_state: Vec<ViBaKeyframe> = keyframes.to_vec();
    let mut xyz: Vec<Vec3F64> = points.to_vec();
    let mut lambda = params.initial_lambda;
    let mut converged = false;
    let mut iters_done = 0usize;

    for _iter in 0..params.max_iterations {
        iters_done += 1;

        // ── Allocate normal-equation blocks ───────────────────────────────
        let dim = n_free_kf * KF_DIM;
        let mut a_mat = vec![0.0_f64; dim * dim]; // row-major 15P×15P
        let mut g_kf  = vec![0.0_f64; dim];
        let mut c_blocks = vec![[0.0_f32; 9]; n_free_pt];
        let mut g_point  = vec![[0.0_f32; 3]; n_free_pt];
        let mut b_by_point: Vec<Vec<(usize, [f32; 18])>> = vec![vec![]; n_free_pt];
        let mut cost = 0.0_f64;

        // ── Visual observations ───────────────────────────────────────────
        // The existing residual_and_jacobians returns (r 2-vec, J_pose 2×6, J_point 2×3).
        // J_pose goes into the top-left 6×6 block of A_kf[kli]; C, B, g_point unchanged.
        for obs in observations {
            let kli = kf_local[obs.pose_idx];
            let xli = pt_local[obs.point_idx];
            let se3 = pose_to_se3(&kf_state[obs.pose_idx].pose);
            let (r, j_pose, j_point) =
                residual_and_jacobians(&se3, &xyz[obs.point_idx], obs.pixel, camera);
            cost += 0.5 * (r[0] * r[0] + r[1] * r[1]) as f64;

            if kli >= 0 {
                // Accumulate J_pose^T J_pose into the 6×6 top-left corner of
                // a_mat block (kli*KF_DIM .. kli*KF_DIM+6, same cols).
                let base = kli as usize * KF_DIM;
                for i in 0..6 { for k in 0..6 {
                    a_mat[(base+i)*dim + base+k] +=
                        (j_pose[i]*j_pose[k] + j_pose[6+i]*j_pose[6+k]) as f64;
                }}
                for i in 0..6 {
                    g_kf[base + i] -=
                        (j_pose[i]*r[0] + j_pose[6+i]*r[1]) as f64;
                }
            }
            // C, g_point, b_by_point: identical to existing bundle_adjust_schur
            // (the point-block cross-term only involves the first 6 cols of kf state)
            // ... (copy from existing code) ...
        }

        // ── IMU edges ─────────────────────────────────────────────────────
        for edge in imu_edges {
            let kli = kf_local[edge.from_idx];
            let klj = kf_local[edge.to_idx];
            // Skip if both endpoints are fixed.
            if kli < 0 && klj < 0 { continue; }

            let kf_i = &kf_state[edge.from_idx];
            let kf_j = &kf_state[edge.to_idx];
            let pre   = &edge.preintegrated;
            let dt    = pre.dt;
            let g     = params.gravity;

            // State quantities
            let r_wb_i = kf_i.pose.rotation.transpose(); // body-to-world at i
            let r_wb_j = kf_j.pose.rotation.transpose();
            let r_wc_i = kf_i.pose.rotation;             // = R_wb_i^T
            let p_i    = -(r_wb_i * kf_i.pose.translation);
            let p_j    = -(r_wb_j * kf_j.pose.translation);
            let v_i    = kf_i.velocity;
            let v_j    = kf_j.velocity;

            // Bias-corrected deltas
            let dr = pre.delta_rotation_with_bias(&kf_i.bias);
            let dv = pre.delta_velocity_with_bias(&kf_i.bias);
            let dp = pre.delta_position_with_bias(&kf_i.bias);

            // Residuals
            // r_R: 3-vec
            let r_r_mat = dr.transpose() * (r_wc_i * r_wb_j);
            let r_r = so3_log(r_r_mat); // implement SO3 log
            // r_v: 3-vec
            let b_vec = v_j - v_i - g * dt;
            let r_v   = r_wc_i * b_vec - dv;
            // r_p: 3-vec
            let a_vec = p_j - p_i - v_i * dt - g * (0.5 * dt * dt);
            let r_p   = r_wc_i * a_vec - dp;

            // Stack residual: r_imu = [r_R(3), r_v(3), r_p(3)]
            let r_imu: [f64; 9] = [
                r_r.x, r_r.y, r_r.z,
                r_v.x, r_v.y, r_v.z,
                r_p.x, r_p.y, r_p.z,
            ];
            cost += 0.5 * r_imu.iter().map(|x| x*x).sum::<f64>(); // unweighted cost; use Ω below

            // Information matrix Ω = covariance^{-1} (9×9, column-major)
            let omega = invert_9x9(&pre.covariance); // you need to implement this

            // Right-Jacobian inverse at r_R
            let jr_inv = so3_right_jacobian_inverse(r_r); // 3×3

            // Bias Jacobians from preintegration
            let dr_dbg = pre.d_rotation_d_bias_gyro;  // 3×3
            let dv_dbg = pre.d_velocity_d_bias_gyro;  // 3×3
            let dv_dba = pre.d_velocity_d_bias_accel; // 3×3
            let dp_dbg = pre.d_position_d_bias_gyro;  // 3×3
            let dp_dba = pre.d_position_d_bias_accel; // 3×3

            // Assemble J_i (9×15) and J_j (9×15).
            // Column layout: [ρ(0:3), ω(3:6), v(6:9), b_g(9:12), b_a(12:15)]
            // Row layout:    [r_R(0:3), r_v(3:6), r_p(6:9)]
            //
            // J_i:
            //   r_R row: ρ→0, ω→ Jr_inv·Rj_wc^T (=Jr_inv·Rwb_j), v→0, bg→−Jr_inv·ΔR^T·dr_dbg, ba→0
            //   r_v row: ρ→0, ω→skew(r_v+dv),    v→−R_wc_i,       bg→−dv_dbg, ba→−dv_dba
            //   r_p row: ρ→R_wc_i, ω→skew(r_p+dp), v→−R_wc_i·dt,  bg→−dp_dbg, ba→−dp_dba
            //
            // J_j:
            //   r_R row: ρ→0, ω→−Jr_inv·Rwb_j,  rest→0
            //   r_v row: ρ→0, ω→0, v→+R_wc_i, rest→0
            //   r_p row: ρ→−R_wc_i, rest→0

            // Build J_i as flat 9×15 row-major (or column-major — be consistent)
            // then compute:
            //   A_kf[i,i] += J_i^T Ω J_i      g_kf[i] -= J_i^T Ω r_imu
            //   A_kf[j,j] += J_j^T Ω J_j      g_kf[j] -= J_j^T Ω r_imu
            //   A_kf[i,j] += J_i^T Ω J_j      A_kf[j,i] = A_kf[i,j]^T

            // ... (implement the accumulation) ...
            let _ = (kli, klj, jr_inv, omega, dr, dr_dbg, dv_dbg, dv_dba, dp_dbg, dp_dba);
        }

        // ── LM damping ────────────────────────────────────────────────────
        for k in 0..n_free_kf * KF_DIM {
            a_mat[k * dim + k] += lambda;
        }
        for cb in &mut c_blocks {
            for d in 0..3 { cb[d*3+d] += lambda as f32; }
        }

        // ── Schur eliminate points: M = A_kf − B·C⁻¹·B^T ────────────────
        // (identical logic to existing bundle_adjust_schur_with_priors,
        //  but operating on dim=15P instead of 6P)
        let mut m_mat = Mat::<f64>::zeros(dim, dim);
        let mut m_vec = vec![0.0_f64; dim];
        // ... copy Schur fill from existing code ...

        // ── Cholesky solve ────────────────────────────────────────────────
        // symmetrise M first (same as existing code)
        let chol = m_mat.llt(faer::Side::Lower)
            .map_err(|e| ViBaError::CholeskyFailed(format!("{e:?}")))?;
        let rhs = Mat::<f64>::from_fn(dim, 1, |i, _| m_vec[i]);
        let d_kf_col = chol.solve(&rhs);

        // ── Back-substitute points ────────────────────────────────────────
        // (identical to existing code; δ_pose comes from first 6 elements of each
        //  15-block in d_kf_col; point back-substitution uses B^T · δ_pose)

        // ── Retract ───────────────────────────────────────────────────────
        for i_global in 0..nkf {
            let kli = kf_local[i_global];
            if kli < 0 { continue; }
            let base = kli as usize * KF_DIM;
            let delta_pose: [f32; 6] = std::array::from_fn(|k| d_kf_col[(base+k, 0)] as f32);
            let se3 = pose_to_se3(&kf_state[i_global].pose);
            kf_state[i_global].pose = se3_to_pose(&se3.retract(&delta_pose));
            kf_state[i_global].velocity.x += d_kf_col[(base+6, 0)];
            kf_state[i_global].velocity.y += d_kf_col[(base+7, 0)];
            kf_state[i_global].velocity.z += d_kf_col[(base+8, 0)];
            kf_state[i_global].bias.gyro.x += d_kf_col[(base+9,  0)];
            kf_state[i_global].bias.gyro.y += d_kf_col[(base+10, 0)];
            kf_state[i_global].bias.gyro.z += d_kf_col[(base+11, 0)];
            kf_state[i_global].bias.accel.x += d_kf_col[(base+12, 0)];
            kf_state[i_global].bias.accel.y += d_kf_col[(base+13, 0)];
            kf_state[i_global].bias.accel.z += d_kf_col[(base+14, 0)];
        }

        // ... LM accept/reject cost comparison, lambda update, convergence check ...
    }

    Ok(ViBaResult {
        keyframes: kf_state,
        points: xyz,
        iterations: iters_done,
        converged,
        final_cost: 0.0,
    })
}
```

---

## Part 7 — Utility functions needed

### 7.1 `so3_log` — SO(3) logarithm

Inverse of `SO3F64::exp`.  Returns the rotation vector `ω` such that `Exp(ω) = R`.

```rust
fn so3_log(r: Mat3F64) -> Vec3F64 {
    let trace = r.col(0).x + r.col(1).y + r.col(2).z;
    let cos_angle = ((trace - 1.0) * 0.5).clamp(-1.0, 1.0);
    let angle = cos_angle.acos();
    if angle.abs() < 1e-8 {
        // Small angle: use first-order approximation
        return Vec3F64::new(
            r.col(1).z - r.col(2).y,
            r.col(2).x - r.col(0).z,
            r.col(0).y - r.col(1).x,
        ) * 0.5;
    }
    let s = angle / (2.0 * angle.sin());
    Vec3F64::new(
        (r.col(1).z - r.col(2).y) * s,
        (r.col(2).x - r.col(0).z) * s,
        (r.col(0).y - r.col(1).x) * s,
    )
}
```

Reference: slambook-en Ch. 4, eq. 4.22; Solà §6.2.

### 7.2 `so3_right_jacobian_inverse` — `Jr^{-1}(ω)`

Used in the rotation residual Jacobians.

```rust
fn so3_right_jacobian_inverse(omega: Vec3F64) -> Mat3F64 {
    let theta = (omega.x*omega.x + omega.y*omega.y + omega.z*omega.z).sqrt();
    let skew = skew3(omega);
    let skew2 = skew * skew; // skew · skew (Mat3×Mat3)
    if theta < 1e-5 {
        // First-order: Jr^{-1} ≈ I + ½[ω]_×
        return Mat3F64::IDENTITY + skew * 0.5;
    }
    let c = (1.0 / (theta * theta))
          - (1.0 + theta.cos()) / (2.0 * theta * theta.sin());
    Mat3F64::IDENTITY + skew * 0.5 + skew2 * c
}
```

Reference: Forster eq. A.8; Solà §6, eq. (99).

### 7.3 `skew3` — skew-symmetric matrix

```rust
fn skew3(v: Vec3F64) -> Mat3F64 {
    Mat3F64::from_cols(
        Vec3F64::new(  0.0,  v.z, -v.y),
        Vec3F64::new( -v.z,  0.0,  v.x),
        Vec3F64::new(  v.y, -v.x,  0.0),
    )
}
```

### 7.4 `invert_9x9` — 9×9 matrix inverse

Use `faer` directly rather than hand-rolling:

```rust
fn invert_9x9(m: &[f64; 81]) -> [f64; 81] {
    let mat = Mat::<f64>::from_fn(9, 9, |r, c| m[r + c * 9]); // column-major input
    let inv = mat.partial_piv_lu().inverse();
    let mut out = [0.0_f64; 81];
    for r in 0..9 { for c in 0..9 { out[r + c * 9] = inv[(r, c)]; } }
    out
}
```

---

## Part 8 — Testing strategy (strongly recommended)

### Step 1: Test `so3_log` against `so3_exp`

```
assert ‖Log(Exp(ω)) − ω‖ < ε   for random ω with ‖ω‖ < π
```

### Step 2: Test IMU residual against `predict`

Given ground-truth consecutive keyframe states (R_i, p_i, v_i) and a
`PreintegratedImu` that correctly integrates the motion between them:
```
r_imu should be ≈ 0 at the true states
```

### Step 3: Finite-difference Jacobian check

For each column `k` of `J_i`:
```
(r_imu(x_i + ε·e_k) − r_imu(x_i)) / ε  ≈  J_i[:, k]
```
This is the most reliable way to catch sign errors in the rotation Jacobians.

### Step 4: Two-keyframe synthetic test

Create two keyframes with known relative motion, add one IMU edge with a
known constant-acceleration measurement.  VI-BA should:
- Recover velocity at both keyframes
- Converge in a small number of iterations

---

## Summary checklist

| Step | What to implement | Where to look |
|---|---|---|
| 1 | `so3_log`, `skew3` | This doc §7; slambook Ch. 4 |
| 2 | `so3_right_jacobian_inverse` | Solà §6; Forster App. A |
| 3 | `invert_9x9` via faer | §7.4 |
| 4 | `imu_residual_and_jacobians()` | §3–4; Forster Table I |
| 5 | IMU block accumulation into `a_mat` | §4.4; §6 scaffold |
| 6 | Extend Schur/Cholesky to 15P dim | §5; existing code |
| 7 | Retract all 15 DOF per keyframe | §6 scaffold |
| 8 | Wire up from `pipeline.rs` | pipeline.rs line 843 |
